<div align="center">

# Training an Interatomic Potential from Scratch
### *Custom `.traj` Dataset · Three Trainers · Full GOAL Infrastructure*

</div>

---

**What this notebook teaches:**

| # | Topic |
|---|-------|
| 1 | Generate a synthetic MD dataset and save it as ASE `.traj` files |
| 2 | Load multiple `.traj` files using GOAL's `TrajectoryDataset` |
| 3 | Build an `InvariantGNN` backbone + custom energy/forces head |
| 4 | Train with **Trainer 1 — MiniTrainer** (pure PyTorch, simplest) |
| 5 | Train with **Trainer 2 — Lightning `Trainer` + `GOALModule`** (production standard) |
| 6 | Train with **Trainer 3 — `FabricTrainer`** (full control + multi-GPU) |
| 7 | Evaluate and compare the three approaches |

**Who this is for:**  
Students and practitioners who want to go beyond toy examples and understand
*every layer* of the GOAL training stack — from raw trajectory files on disk
to a deployed model.

> **Dataset:** We use a **Lennard-Jones potential** to generate synthetic
> molecular-dynamics frames for several small clusters.  This keeps the
> tutorial fully self-contained — no internet access needed, no DFT license
> required.  The workflow is *identical* for real DFT datasets (VASP OUTCARs,
> CP2K outputs, ANI-1ccx, MD17, etc.) — just point `TrajectoryDataset` at
> your `.traj` files.


---

## 1 · Imports

We start by importing everything we will need throughout the notebook.
Notice that GOAL's modules are grouped by their role: data, models, training.


In [ ]:
from __future__ import annotations

import math
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

# ASE -- used only for creating/reading trajectory files
from ase import Atoms
from ase.io import Trajectory

# PyTorch Geometric -- batching
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.utils import scatter

# -- GOAL: data contract -----------------------------------------------
from goal.ml.data.graph import AtomicGraph, NodeFeatures
from goal.ml.data.datasets.trajectory import TrajectoryDataset

# -- GOAL: neural-network building blocks ------------------------------
from goal.ml.nn.models.invariant import InvariantGNN

# -- GOAL: training ----------------------------------------------------
from goal.ml.training.loss import (
    CompositeLoss,
    EnergyLoss,
    ForcesLoss,
    WeightedLoss,
)
from goal.ml.utils.mini_trainer import MiniTrainer, TrainingHistory, graph_step
from goal.ml.utils.fabric_trainer import FabricTrainer, graph_fabric_step

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device : {device}')
print(f'PyTorch      : {torch.__version__}')


---

## 2 · Synthetic Dataset — Lennard-Jones Clusters

### 2.1 Why Lennard-Jones?

The **Lennard-Jones (LJ) potential** is the classic pair potential for
noble-gas clusters.  It has an analytic energy *and* analytic forces, which
means we can generate arbitrarily large datasets instantly and verify our
training by comparing to the exact curve.

$$E_{LJ} = 4\varepsilon \sum_{i < j} \left[
    \left(\frac{\sigma}{r_{ij}}\right)^{12}
  - \left(\frac{\sigma}{r_{ij}}\right)^{6}
\right]$$

Forces are $\mathbf{F}_i = -\nabla_i E$.

We will generate three separate `.traj` files — one per cluster size — and
treat them as independent data sources that GOAL's `TrajectoryDataset` will
load and merge.


In [ ]:
# -- Lennard-Jones parameters ------------------------------------------
EPSILON = 0.0104  # eV  (Ar-Ar well depth)
SIGMA   = 3.40    # Angstrom  (Ar-Ar collision diameter)
CUTOFF  = 8.5     # Angstrom  neighbour cutoff for the model


def lj_energy_forces(positions: np.ndarray) -> tuple[float, np.ndarray]:
    """Compute LJ energy (eV) and forces (eV/A) for an array of positions.

    Parameters
    ----------
    positions : (N, 3) array
        Cartesian coordinates in Angstrom.

    Returns
    -------
    energy : float
    forces : (N, 3) ndarray
    """
    N = len(positions)
    energy = 0.0
    forces = np.zeros_like(positions)

    for i in range(N):
        for j in range(i + 1, N):
            r_vec = positions[j] - positions[i]
            r = np.linalg.norm(r_vec)
            if r < 1e-10:
                continue
            sr6  = (SIGMA / r) ** 6
            sr12 = sr6 ** 2
            energy += 4 * EPSILON * (sr12 - sr6)
            f_mag = 4 * EPSILON * (12 * sr12 - 6 * sr6) / r
            f_vec = f_mag * (r_vec / r)
            forces[i] -= f_vec
            forces[j] += f_vec

    return energy, forces


# Quick sanity check: equilibrium dimer r = 2^(1/6) * sigma => F = 0
r_eq = 2 ** (1 / 6) * SIGMA
pos_dimer = np.array([[0.0, 0.0, 0.0], [r_eq, 0.0, 0.0]])
E_eq, F_eq = lj_energy_forces(pos_dimer)
print(f'Dimer at r_eq = {r_eq:.3f} A')
print(f'  E = {E_eq:.6f} eV  (should be -{EPSILON:.4f} eV)')
print(f'  |F| = {np.linalg.norm(F_eq[0]):.2e} eV/A  (should be ~0)')


### 2.2 Generate MD frames and save as `.traj` files

We run a simple random-walk MD to generate diverse configurations.
Three cluster sizes (5, 8, 13 atoms) are saved to separate `.traj` files inside
`data/lj_clusters/`.  This simulates the typical scenario where you have data
from multiple independent MD runs or multiple chemical systems.


In [ ]:
DATA_DIR = Path('data/lj_clusters')
DATA_DIR.mkdir(parents=True, exist_ok=True)

N_FRAMES     = 300   # frames per cluster size
DISPLACEMENT = 0.25  # Angstrom -- random displacement magnitude per MD step
ATOMIC_NUM   = 18    # Argon


def generate_cluster_traj(
    n_atoms: int,
    n_frames: int,
    filename: Path,
    seed: int = 0,
) -> None:
    """Generate a random-walk trajectory for an n_atoms LJ cluster.

    Starts from a simple cubic layout (atoms well-separated),
    then iterates random displacements, saving every frame.
    Energy and forces from the LJ potential are stored in
    atoms.info['energy'] and atoms.arrays['forces'].
    """
    rng = np.random.default_rng(seed)

    # Simple cubic initial layout, slightly beyond equilibrium
    side  = math.ceil(n_atoms ** (1 / 3))
    grid  = np.array(
        [[i, j, k]
         for i in range(side)
         for j in range(side)
         for k in range(side)],
        dtype=float,
    )[:n_atoms] * SIGMA * 1.1

    positions = grid.copy()
    traj = Trajectory(str(filename), mode='w')

    for _ in range(n_frames):
        positions += rng.normal(scale=DISPLACEMENT, size=positions.shape)
        energy, forces = lj_energy_forces(positions)

        atoms = Atoms(
            numbers=[ATOMIC_NUM] * n_atoms,
            positions=positions,
        )
        # Store energy and forces so TrajectoryDataset can find them
        atoms.info['energy'] = energy
        atoms.arrays['forces'] = forces
        traj.write(atoms)

    traj.close()
    print(f'  Saved {n_frames} frames -> {filename}  ({n_atoms} atoms/frame)')


print('Generating trajectories...')
generate_cluster_traj(5,  N_FRAMES, DATA_DIR / 'cluster5.traj',  seed=1)
generate_cluster_traj(8,  N_FRAMES, DATA_DIR / 'cluster8.traj',  seed=2)
generate_cluster_traj(13, N_FRAMES, DATA_DIR / 'cluster13.traj', seed=3)
print('Done.')


---

## 3 · Loading Data with `TrajectoryDataset`

### Why not just use `torch.utils.data.Dataset` directly?

GOAL's `TrajectoryDataset` does more than just reading files:

1. **Builds the neighbour list** — calls ASE (or matscipy/nvalchemiops) to
   compute all pairs within the cutoff radius.  This is the expensive step
   that you want done once at load time, not inside the training loop.
2. **Converts to `AtomicGraph`** — wraps positions, atomic numbers, edge index,
   edge vectors, and targets in a single PyG `Data` object.
3. **Works with `DataLoader`** — PyG's `DataLoader` automatically batches a
   list of `AtomicGraph` objects into a single mega-graph with a `batch` tensor.

### 3.1 Load each `.traj` file

The `TrajectoryDataset` constructor accepts either:
- A path to a **directory** containing `{split}.traj` files
  (e.g. `root/train.traj`, `root/val.traj`)
- A path to a **single `.traj` file** directly (what we use here)

The `energy_key` and `forces_key` arguments tell it where to find the
labels inside each ASE `Atoms` object (`atoms.info[key]` for scalars,
`atoms.arrays[key]` for per-atom arrays).  If the keys are not found,
it falls back to `atoms.get_potential_energy()` and `atoms.get_forces()`.


In [ ]:
def load_traj(path: Path, cutoff: float) -> TrajectoryDataset:
    """Load a single .traj file as a TrajectoryDataset."""
    return TrajectoryDataset(
        root=path,                   # path to the .traj file directly
        cutoff=cutoff,
        energy_key='energy',         # atoms.info['energy']
        forces_key='forces',         # atoms.arrays['forces']
        dtype=torch.float32,
        neighbor_list_backend='ase', # 'matscipy' or 'nvalchemiops' on GPU
    )


print(f'Loading datasets with cutoff = {CUTOFF} A ...')
ds5  = load_traj(DATA_DIR / 'cluster5.traj',  CUTOFF)
ds8  = load_traj(DATA_DIR / 'cluster8.traj',  CUTOFF)
ds13 = load_traj(DATA_DIR / 'cluster13.traj', CUTOFF)

print(f'  cluster5  : {len(ds5):>4} graphs')
print(f'  cluster8  : {len(ds8):>4} graphs')
print(f'  cluster13 : {len(ds13):>4} graphs')
print(f'  Total     : {len(ds5) + len(ds8) + len(ds13):>4} graphs')

# Inspect one sample
sample = ds5[0]
print(f'\nSample graph from cluster5:')
print(f'  num_atoms   = {sample.num_atoms}')
print(f'  edge_index  shape = {sample.edge_index.shape}  ({sample.edge_index.shape[1]} edges)')
print(f'  energy      = {sample.energy.item():.4f} eV')
print(f'  forces shape      = {sample.forces.shape}')


### 3.2 Split and create DataLoaders

We merge all three datasets, shuffle, and split 70 / 15 / 15.
PyG's `DataLoader` handles the batching automatically — it stacks all the
individual graphs in a batch into one *mega-graph* and adds a `batch` tensor
that maps each atom back to its original graph.


In [ ]:
from torch.utils.data import ConcatDataset, random_split

# Merge all three datasets
all_graphs = ConcatDataset([ds5, ds8, ds13])
N_TOTAL    = len(all_graphs)

n_train = int(0.70 * N_TOTAL)
n_val   = int(0.15 * N_TOTAL)
n_test  = N_TOTAL - n_train - n_val

train_ds, val_ds, test_ds = random_split(
    all_graphs,
    [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42),
)
print(f'Split: {n_train} train | {n_val} val | {n_test} test')

BATCH_SIZE = 16
train_loader = PyGDataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = PyGDataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = PyGDataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

# Inspect a batch
batch = next(iter(train_loader))
print(f'\nSample batch:')
print(f'  num_graphs       = {batch.num_graphs}')
print(f'  pos.shape        = {batch.pos.shape}   (N_total x 3)')
print(f'  edge_index.shape = {batch.edge_index.shape}  (2 x E_total)')
print(f'  energy.shape     = {batch.energy.shape}')
print(f'  forces.shape     = {batch.forces.shape}')
print(f'  batch.shape      = {batch.batch.shape}  (atom -> graph mapping)')


---

## 4 · Building the Model

### 4.1 Architecture overview

We build a **modular** model with two components:

```
AtomicGraph
    |
    v
+-----------------------------------------------------------+
|  InvariantGNN  (backbone)                                 |
|                                                           |
|  z  --[Embedding]--> h_0       learnable atom-type vecs  |
|                                                           |
|  for L interaction layers:                                |
|    message: m_ij = MLP([h_i || h_j || phi(r_ij)])        |
|    update:  h_i  = h_i + sum_j m_ij                      |
|                                                           |
|  output: NodeFeatures  (N, hidden_channels)               |
+-----------------------------------------------------------+
    |
    v  NodeFeatures
+-----------------------------------------------------------+
|  EnergyForcesHead  (output head)                         |
|                                                           |
|  h_i --[MLP]--> e_i (atomic energy contribution)        |
|  E_total = sum_i e_i                                      |
|  F_i     = -dE_total / dr_i   (autograd)                 |
+-----------------------------------------------------------+
```

### Why atomic energy decomposition?

This approach — called **energy decomposition** or the *Behler-Parrinello*
ansatz — is the cornerstone of modern interatomic potentials:

- Each atom contributes an *atomic energy* $e_i$ that depends only on its
  local chemical environment (atoms within the cutoff sphere).
- Total energy: $E = \sum_i e_i$ → scalar, size-extensive.
- Forces are *exact* gradients: $\mathbf{F}_i = -\nabla_i E$ via autograd.
  This guarantees **energy conservation** in MD simulations.

Compare this to the diatomic tutorial where we summed *edge* energies.  For
multi-component systems with variable sizes, summing *atomic* energies is far
more natural — and it is what all production-quality MLIPs do.

### 4.2 The `EnergyForcesHead`


In [ ]:
class EnergyForcesHead(nn.Module):
    """Atomic-energy decomposition head for energy + force prediction.

    Maps backbone node features h_i (shape: N x D) to a per-atom scalar
    energy contribution e_i via a small MLP, then aggregates to per-graph
    total energies.  Forces are computed as -dE/dpos via autograd.

    Parameters
    ----------
    backbone_dim : int
        Width of the node features from the backbone.
    hidden_dim : int
        Width of the two-layer MLP inside the head.
    """

    def __init__(self, backbone_dim: int, hidden_dim: int = 64) -> None:
        super().__init__()
        # Small MLP: backbone_dim -> hidden_dim -> 1
        # The final linear has no bias to keep atomic energies purely
        # determined by the local environment (no global offset).
        self.mlp = nn.Sequential(
            nn.Linear(backbone_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1, bias=False),
        )

    @property
    def output_keys(self) -> list[str]:
        return ['energy', 'forces', 'num_atoms']

    def forward(
        self,
        features: NodeFeatures,
        graph: AtomicGraph,
    ) -> dict[str, torch.Tensor]:
        # Enable gradient tracking on positions (needed for forces via autograd)
        graph.pos.requires_grad_(True)

        # Per-atom scalar energy contributions
        h = features.node_feats             # (N, backbone_dim)
        e_atomic = self.mlp(h).squeeze(-1)  # (N,)

        # Aggregate to per-graph total energy
        batch = (
            graph.batch
            if graph.batch is not None
            else torch.zeros(graph.num_atoms, dtype=torch.long, device=graph.pos.device)
        )
        num_graphs = int(batch.max().item()) + 1

        energy = scatter(
            e_atomic, batch, dim=0, reduce='sum', dim_size=num_graphs
        )  # (B,) -- one total energy per structure in the batch

        # Forces via autograd: F_i = -dE/dr_i
        # We sum over all graphs so we get forces for every atom at once.
        # create_graph=True during training allows backprop through the forces.
        grad = torch.autograd.grad(
            outputs=energy.sum(),
            inputs=graph.pos,
            create_graph=self.training,
            retain_graph=True,
        )[0]
        forces = -grad  # (N, 3)

        # Count atoms per graph (needed by EnergyLoss for per-atom normalisation)
        num_atoms = scatter(
            torch.ones(graph.num_atoms, device=graph.pos.device),
            batch, dim=0, reduce='sum', dim_size=num_graphs,
        )  # (B,)

        return {'energy': energy, 'forces': forces, 'num_atoms': num_atoms}


### 4.3 Assemble backbone + head

We use GOAL's `InvariantGNN` as the backbone.  It runs several layers of
message passing using the pre-built neighbour list stored in the graph.


In [ ]:
HIDDEN_DIM = 64
NUM_LAYERS = 3
NUM_RADIAL = 8


def build_model() -> nn.Module:
    """Construct a fresh backbone + head as a single nn.Module."""

    backbone = InvariantGNN(
        num_elements    = 120,         # full periodic table
        embedding_dim   = 32,          # atom-type embedding size
        hidden_channels = HIDDEN_DIM,
        num_interactions= NUM_LAYERS,
        cutoff          = CUTOFF,
        num_radial_basis= NUM_RADIAL,
    )

    head = EnergyForcesHead(
        backbone_dim = HIDDEN_DIM,
        hidden_dim   = 64,
    )

    class FullModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.backbone = backbone
            self.head     = head

        @property
        def output_keys(self):
            return self.head.output_keys

        def forward(self, graph):
            features = self.backbone(graph)
            return self.head(features, graph)

    return FullModel()


model_ref = build_model()
n_params  = sum(p.numel() for p in model_ref.parameters())
print(f'Total parameters: {n_params:,}')

# Quick forward pass to verify shapes
model_ref.eval()
with torch.enable_grad():
    out = model_ref(batch)

print(f"energy shape : {out['energy'].shape}   (B,)")
print(f"forces shape : {out['forces'].shape}  (N_total, 3)")
print(f"num_atoms    : {out['num_atoms'].tolist()[:4]} ...")


---

## 5 · Loss Function

We use a **composite loss** that mixes energy and force errors:

$$\mathcal{L} = w_E \cdot \text{MAE}\!\left(\frac{E_\text{pred}}{N},\,
\frac{E_\text{true}}{N}\right)
+ w_F \cdot \text{MAE}\!\left(\mathbf{F}_\text{pred},\, \mathbf{F}_\text{true}\right)$$

Dividing by $N$ (number of atoms) makes the energy loss **size-invariant** —
a cluster of 13 atoms is not penalised 2.6× more than one of 5 atoms just
because it is larger.

We weight forces more heavily (`forces_weight=10`) because:
- Forces provide $3N$ training signals vs. 1 energy per structure.
- Correct forces are essential for using the potential in MD simulations.

The `CompositeLoss` takes a list of `WeightedLoss` wrappers.  Each wrapped
loss receives the full predictions dict and the batch (an `AtomicGraph`).
`EnergyLoss` reads `batch.energy` and `EnergyLoss` reads `batch.forces`
directly from the graph object.


In [ ]:
def build_loss(
    energy_weight: float = 1.0,
    forces_weight: float = 10.0,
) -> CompositeLoss:
    """Build a composite energy + forces MAE loss."""
    return CompositeLoss([
        WeightedLoss(
            EnergyLoss(loss_fn='mae'),
            weight=energy_weight,
            label='energy_loss',
        ),
        WeightedLoss(
            ForcesLoss(loss_fn='mae'),
            weight=forces_weight,
            label='forces_loss',
        ),
    ])


# The CompositeLoss returns a dict:
#   {'total': tensor, 'energy_loss': tensor, 'forces_loss': tensor}
# The 'total' key is the weighted sum -- this is what each trainer optimises.
criterion = build_loss(energy_weight=1.0, forces_weight=10.0)
print('Loss function:', criterion)


---

## 6 · Trainer 1 — `MiniTrainer`

### What is the MiniTrainer?

`MiniTrainer` is the **simplest** of the three trainers.  It is a plain
Python `for` loop wrapped in a class, with no Lightning, no Hydra, and no
distributed training.  It runs on a **single device** (CPU or one GPU).

**When to use it:**
- Quick experiments in a Jupyter notebook
- Debugging model architecture or loss functions
- Teaching/demos where simplicity matters

**When NOT to use it:**
- Multi-GPU or multi-node training → use FabricTrainer or GOALModule
- Production runs with disk checkpointing, callbacks, EMA → use GOALModule

### How it works

```
MiniTrainer.fit(train_loader, val_loader, epochs)
    |
    for epoch in range(epochs):
    |   for batch in train_loader:           <- training
    |       optimizer.zero_grad()
    |       result = step_fn(batch, model, loss_fn, device)
    |       loss   = result['total']
    |       loss.backward()
    |       clip_grad_norm_(...)
    |       optimizer.step()
    |
    |   with torch.enable_grad():            <- validation (enable_grad for forces)
    |       result = step_fn(batch, model, loss_fn, device)
    |
    |   scheduler.step()
    |   history.append(train_loss, val_loss)
```

### The `graph_step` function

Since our batches are `AtomicGraph` objects (not plain `(x, y)` tensors),
we use the built-in `graph_step` function:

```python
def graph_step(batch, model, loss_fn, device):
    batch       = batch.to(device)
    predictions = model(batch)
    losses      = loss_fn(predictions, batch)  # reads .energy, .forces from batch
    return losses   # dict: {'total', 'energy_loss', 'forces_loss'}
```

> **Important:** Our head calls `graph.pos.requires_grad_(True)` inside
> `forward()` to enable force computation via autograd.  This requires that
> `torch.enable_grad()` is active.  GOAL's `graph_step` wraps the call
> appropriately, so forces work correctly both during training and validation.


In [ ]:
# Build a fresh model for this trainer
model_mini     = build_model()
criterion_mini = build_loss()

optimizer_mini = torch.optim.AdamW(
    model_mini.parameters(),
    lr=3e-3,
    weight_decay=1e-4,
)
scheduler_mini = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_mini,
    T_max=30,
    eta_min=1e-5,
)

mini_trainer = MiniTrainer(
    model    = model_mini,
    loss_fn  = criterion_mini,
    optimizer= optimizer_mini,
    scheduler= scheduler_mini,
    device   = 'auto',       # picks CUDA if available, else CPU
    step_fn  = graph_step,   # AtomicGraph-aware step function
    grad_clip= 1.0,          # clip gradient norm
    enable_progress = True,
)

print(f'MiniTrainer device: {mini_trainer.device}')
print(f'Parameters       : {sum(p.numel() for p in model_mini.parameters()):,}')


In [ ]:
print('Training with MiniTrainer ...')
history_mini = mini_trainer.fit(
    train_loader,
    val_loader              = val_loader,
    epochs                  = 30,
    early_stopping_patience = 10,
    checkpoint_best         = True,  # keep best weights in memory
    verbose                 = True,
)

print(f'\nBest val loss : {history_mini.best_val_loss:.6f} (epoch {history_mini.best_epoch + 1})')
history_mini.plot()


---

## 7 · Trainer 2 — `GOALModule` + Lightning `Trainer`

### What is the Lightning Trainer?

`GOALModule` is a `LightningModule` — it wraps your model, loss, optimiser,
and scheduler in the standard Lightning training protocol.  The Lightning
`Trainer` then handles everything automatically:

| Feature | Handled automatically |
|---------|----------------------|
| Multi-GPU (DDP, FSDP) | Yes |
| Mixed precision (bf16, fp16) | Yes |
| Gradient accumulation | Yes |
| Logging (TensorBoard, W&B, CSV) | Yes |
| Callbacks (early stopping, checkpointing, EMA) | Yes |
| Progress bars | Yes |
| Profiling | Yes |

**When to use it:**
- Production training runs (especially multi-GPU)
- When you want callbacks (model checkpointing, EMA, LR monitoring)
- When you want structured logging (TensorBoard, W&B, MLflow)
- When you use Hydra configs for experiment management

**When NOT to use it:**
- Quick notebook experiments where MiniTrainer is simpler
- When you need unconventional training logic (GAN, RL, curriculum learning)
  → use FabricTrainer instead

### How `GOALModule` works internally

```python
class GOALModule(L.LightningModule):

    def training_step(self, batch, batch_idx):
        # batch IS the AtomicGraph
        predictions = self(batch)              # forward: backbone -> head
        losses      = self.loss(predictions, batch)  # CompositeLoss reads batch.energy etc.
        self.log_dict({'train/total': losses['total'], ...})
        return losses['total']

    def validation_step(self, batch, batch_idx):
        predictions = self(batch)
        losses      = self.loss(predictions, batch)
        self.log_dict({'val/total': losses['total'], ...})

    def configure_optimizers(self):
        # Reads from self.config.training.optimizer
        return {'optimizer': ..., 'lr_scheduler': ...}
```

### Setting up a minimal config

`GOALModule` normally reads its configuration from a Hydra `DictConfig`.
We build one by hand using `OmegaConf.create()` from a plain Python dict.
In a real project you would use `python src/train.py model=invariant_gnn ...`
with a YAML config file.


In [ ]:
import lightning as L
from omegaconf import OmegaConf

from goal.ml.training.module import GOALModule

# Build a minimal OmegaConf config for GOALModule
cfg = OmegaConf.create({
    'training': {
        'optimizer': {
            'lr'            : 3e-3,
            'weight_decay'  : 1e-4,
            'amsgrad'       : True,
            'scheduler_type': 'cosine',  # 'cosine' or 'plateau' or 'none'
            'min_lr'        : 1e-5,
        },
        'max_epochs'       : 30,
        'gradient_clip_val': 1.0,
        'ema': {'enabled': False},  # EMA disabled for brevity
    }
})

# Build model and loss
model_lgtn    = build_model()
criterion_lgtn= build_loss()

# Wrap in GOALModule
goal_module = GOALModule(
    backbone      = model_lgtn.backbone,
    head          = model_lgtn.head,
    loss          = criterion_lgtn,
    config        = cfg,
    compile_model = False,  # set True for torch.compile speed-up (PyTorch 2.0+)
)

print('GOALModule created.')
print(f'  Parameters: {sum(p.numel() for p in goal_module.parameters()):,}')


In [ ]:
# Configure the Lightning Trainer
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

log_dir = Path('logs/lightning_trainer')
log_dir.mkdir(parents=True, exist_ok=True)

trainer = L.Trainer(
    max_epochs          = 30,
    accelerator         = 'auto',     # 'gpu' if CUDA, else 'cpu'
    devices             = 1,
    gradient_clip_val   = 1.0,
    logger              = CSVLogger(str(log_dir), name='lj_experiment'),
    callbacks           = [
        EarlyStopping(
            monitor  = 'val/total',
            patience = 10,
            mode     = 'min',
            verbose  = True,
        ),
        ModelCheckpoint(
            monitor    = 'val/total',
            mode       = 'min',
            save_top_k = 1,
            dirpath    = str(log_dir / 'checkpoints'),
            filename   = 'best-{epoch:02d}',
        ),
    ],
    enable_progress_bar = True,
    log_every_n_steps   = 5,
)

print('Lightning Trainer ready.')


In [ ]:
print('Training with Lightning Trainer + GOALModule ...')
trainer.fit(
    goal_module,
    train_dataloaders = train_loader,
    val_dataloaders   = val_loader,
)

# Read the CSV log and plot loss curves
import pandas as pd

metrics_paths = sorted(log_dir.glob('lj_experiment/version_*/metrics.csv'))
if metrics_paths:
    metrics_df = pd.read_csv(metrics_paths[-1])
    fig, ax = plt.subplots(figsize=(10, 4))
    train_rows = metrics_df.dropna(subset=['train/total'])
    val_rows   = metrics_df.dropna(subset=['val/total'])
    ax.plot(train_rows['epoch'], train_rows['train/total'], label='train total')
    ax.plot(val_rows['epoch'],   val_rows['val/total'],     label='val total')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Composite loss')
    ax.set_title('Lightning Trainer -- Loss Curves')
    ax.legend()
    ax.set_yscale('log')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


---

## 8 · Trainer 3 — `FabricTrainer`  *(Full Control + Multi-GPU)*

### What is Lightning Fabric?

`lightning.Fabric` is the **low-level engine** that powers the Lightning
`Trainer` internally.  `FabricTrainer` exposes it directly, so you get:

- Everything the Lightning Trainer handles automatically
  (DDP, FSDP, mixed precision, gradient clipping, distributed logging)
- But you *write the training loop yourself*

Think of it as a spectrum:

```
MiniTrainer          FabricTrainer           GOALModule + Trainer
------------         ---------------         ----------------------
Pure PyTorch         Lightning Fabric         Lightning Trainer
Single device        Multi-GPU ready          Multi-GPU ready
Loop by hand         Loop by hand             Loop managed for you
No callbacks         No callbacks             Full callbacks
Notebook-friendly    Notebook-friendly        Config-driven (Hydra)
```

### Why would you choose FabricTrainer?

1. **You need multi-GPU** but the Lightning Trainer's abstractions get in
   your way (e.g. custom multi-optimiser steps, GAN training, RL).
2. **You want to log something non-standard** — every batch, per-element
   energies, custom metrics — without fighting Lightning's logging API.
3. **You are transitioning** from a plain PyTorch loop to distributed training
   and want to do it step by step.
4. **You are debugging distributed code** — Fabric makes it easy to test
   the same loop on 1 GPU before scaling to 8.

### Key concepts in Fabric

```python
import lightning

# 1. Create the Fabric object with your hardware config
fabric = lightning.Fabric(
    accelerator = 'gpu',           # or 'cpu', 'mps', 'auto'
    strategy    = 'ddp',           # or 'fsdp', 'deepspeed', 'auto'
    devices     = 2,               # number of GPUs
    precision   = 'bf16-mixed',    # mixed precision
)
fabric.launch()  # initialises the distributed process group

# 2. Wrap model and optimizer -- now they work on any device/strategy
model, optimizer = fabric.setup(model, optimizer)

# 3. Wrap DataLoaders -- they are automatically sharded across GPUs
train_loader = fabric.setup_dataloaders(train_loader)

# 4. In the loop: use fabric.backward() instead of loss.backward()
fabric.backward(loss)  # handles mixed-precision scaling + gradient sync

# 5. Clip gradients through Fabric
fabric.clip_gradients(model, optimizer, max_norm=1.0)

# 6. Only rank 0 should print/save checkpoints
if fabric.is_global_zero:
    torch.save(model.state_dict(), 'checkpoint.pt')
```

### Mixed precision: `bf16-mixed`

On modern NVIDIA GPUs (Ampere: A100, RTX 3090, and newer) and recent AMD
GPUs, **bfloat16** is the preferred mixed-precision format:

- Same dynamic range as float32 (8 exponent bits vs. float16's 5)
- No loss scaling needed (unlike fp16 which can underflow)
- ~2× memory saving, ~2× throughput on Tensor Cores
- Minimal accuracy loss for most neural network training

With `precision='bf16-mixed'`, Fabric casts forward passes to bf16 while
keeping master weights in float32.  You don't change your model code at all.

### Gradient accumulation

`FabricTrainer` supports gradient accumulation via `grad_accumulation_steps`.
Setting it to 4 is equivalent to 4× larger effective batch size with
the same memory footprint:

```
effective_batch = batch_size x grad_accumulation_steps
```

Fabric's `no_backward_sync` context suppresses the expensive all-reduce
communication on accumulation steps, making this efficient in DDP.


In [ ]:
# Build a fresh model for FabricTrainer
model_fab     = build_model()
criterion_fab = build_loss()

optimizer_fab = torch.optim.AdamW(
    model_fab.parameters(),
    lr=3e-3,
    weight_decay=1e-4,
)
scheduler_fab = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_fab, T_max=30, eta_min=1e-5,
)

# FabricTrainer configuration:
#
#   accelerator = 'auto'      -> GPU if available, else CPU
#   strategy    = 'auto'      -> DDP if multiple GPUs, else single-device
#   precision   = None        -> float32  (default, most compatible)
#               = 'bf16-mixed' -> bfloat16 on Ampere+ GPU
#               = '16-mixed'  -> float16 on older GPU (needs loss scaling)
#               = '64-true'   -> float64 for high-precision science
#
#   grad_accumulation_steps = 1  -> set > 1 to simulate larger batches
#
# To run on 2 GPUs: devices=2, strategy='ddp'

fabric_trainer = FabricTrainer(
    model        = model_fab,
    loss_fn      = criterion_fab,
    optimizer    = optimizer_fab,
    train_loader = train_loader,
    val_loader   = val_loader,
    accelerator  = 'auto',
    strategy     = 'auto',
    devices      = 1,
    precision    = None,              # float32
    scheduler    = scheduler_fab,
    step_fn      = graph_fabric_step, # AtomicGraph-aware Fabric step
    grad_clip    = 1.0,
    grad_accumulation_steps = 1,
    enable_progress = True,
)

print('FabricTrainer ready.')
print(f'  Fabric device  : {fabric_trainer.fabric.device}')
print(f'  Global rank    : {fabric_trainer.fabric.global_rank}')
print(f'  World size     : {fabric_trainer.fabric.world_size}')


In [ ]:
print('Training with FabricTrainer ...')
history_fab = fabric_trainer.fit(
    epochs                  = 30,
    early_stopping_patience = 10,
    checkpoint_best         = True,
    verbose                 = True,
)

print(f'\nBest val loss : {history_fab.best_val_loss:.6f} (epoch {history_fab.best_epoch + 1})')
history_fab.plot()


### 8.1 Standalone Fabric — writing the loop yourself

`FabricTrainer` is convenient, but sometimes you need to write every line
yourself.  Here is the absolute minimal Fabric training loop so you understand
exactly what `FabricTrainer` does under the hood.

This is the pattern you would expand for custom multi-optimiser setups,
curriculum learning, or anything non-standard.


In [ ]:
import lightning

# Create a Fabric object -- this is the only Fabric-specific line in setup
fabric_raw = lightning.Fabric(accelerator='auto', devices=1)
fabric_raw.launch()

# Model, optimiser, data
model_raw = build_model()
optim_raw = torch.optim.AdamW(model_raw.parameters(), lr=3e-3)
loss_raw  = build_loss()

# Fabric wraps model + optimizer so they move to the right device/dtype
model_raw, optim_raw = fabric_raw.setup(model_raw, optim_raw)

# Fabric wraps DataLoaders -- in DDP this shards data across GPUs automatically
train_ldr_raw = fabric_raw.setup_dataloaders(
    PyGDataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
)

# The training loop -- 5 epochs to demonstrate the pattern
model_raw.train()
for epoch in range(5):
    epoch_loss = 0.0
    for batch in train_ldr_raw:
        optim_raw.zero_grad()

        # Forward pass (enable_grad is active by default here)
        preds  = model_raw(batch)
        losses = loss_raw(preds, batch)

        # IMPORTANT: use fabric.backward() not loss.backward()
        # It handles mixed-precision loss scaling and gradient synchronisation
        fabric_raw.backward(losses['total'])

        # Clip gradients through Fabric (works correctly in distributed mode)
        fabric_raw.clip_gradients(model_raw, optim_raw, max_norm=1.0)
        optim_raw.step()
        epoch_loss += losses['total'].item()

    mean_loss = epoch_loss / len(train_ldr_raw)

    # Only rank 0 prints (avoids duplicated output on multi-GPU)
    if fabric_raw.is_global_zero:
        print(f'  Epoch {epoch + 1}/5  train_loss = {mean_loss:.4f}')

print('Standalone Fabric loop complete.')


---

## 9 · Evaluation and Comparison

Let's compare the three trained models on the held-out test set.
We measure:
- **Energy MAE per atom** (eV/atom) — per-atom normalised for fair comparison
  across cluster sizes
- **Forces MAE** (eV/Å) — mean absolute error on all force components


In [ ]:
def evaluate_on_test(
    model: nn.Module,
    loader,
    device: torch.device,
) -> dict:
    """Compute mean absolute errors on energy and forces."""
    model.eval().to(device)
    e_errors, f_errors = [], []

    for batch in loader:
        batch = batch.to(device)
        with torch.enable_grad():  # needed for force computation via autograd
            preds = model(batch)

        # Energy MAE per atom
        bv        = batch.batch
        ng        = int(bv.max().item()) + 1
        num_atoms = scatter(
            torch.ones(batch.num_atoms, device=device),
            bv, dim=0, reduce='sum', dim_size=ng,
        )
        e_pred_pa = (preds['energy'] / num_atoms).detach().cpu().numpy()
        e_true_pa = (batch.energy   / num_atoms).detach().cpu().numpy()
        e_errors.extend(np.abs(e_pred_pa - e_true_pa).tolist())

        # Forces MAE (all components)
        f_pred = preds['forces'].detach().cpu().numpy()
        f_true = batch.forces.detach().cpu().numpy()
        f_errors.extend(np.abs(f_pred - f_true).flatten().tolist())

    return {
        'energy_mae_pa': float(np.mean(e_errors)),
        'forces_mae'   : float(np.mean(f_errors)),
    }


test_device = torch.device('cpu')  # evaluate on CPU for reproducibility

# Restore best MiniTrainer weights
if mini_trainer._best_state is not None:
    model_mini.load_state_dict(mini_trainer._best_state)

# GOALModule: Lightning has already restored best weights via ModelCheckpoint
class _LgtnWrapper(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, g): return self.m(g)
model_lgtn_eval = _LgtnWrapper(goal_module)

# Restore best FabricTrainer weights
raw_fab = (
    fabric_trainer.model.module
    if hasattr(fabric_trainer.model, 'module')
    else fabric_trainer.model
)
if fabric_trainer._best_state is not None:
    raw_fab.load_state_dict(fabric_trainer._best_state)

results = {
    'MiniTrainer'     : evaluate_on_test(model_mini,       test_loader, test_device),
    'LightningTrainer': evaluate_on_test(model_lgtn_eval,  test_loader, test_device),
    'FabricTrainer'   : evaluate_on_test(raw_fab,          test_loader, test_device),
}

print(f"{'Trainer':<20} {'Energy MAE/atom (eV)':>22} {'Forces MAE (eV/A)':>20}")
print('-' * 66)
for name, r in results.items():
    print(f"{name:<20} {r['energy_mae_pa']:>22.6f} {r['forces_mae']:>20.6f}")


In [ ]:
# Visual comparison: predicted vs. true energies on the test set
fig, axes = plt.subplots(1, 3, figsize=(15, 5))


def scatter_energy(ax, model, loader, device, title):
    model.eval().to(device)
    e_pred, e_true = [], []
    for batch in loader:
        batch = batch.to(device)
        with torch.enable_grad():
            out = model(batch)
        bv = batch.batch
        ng = int(bv.max().item()) + 1
        na = scatter(
            torch.ones(batch.num_atoms, device=device),
            bv, dim=0, reduce='sum', dim_size=ng,
        )
        e_pred.extend((out['energy'] / na).detach().cpu().numpy().tolist())
        e_true.extend((batch.energy  / na).detach().cpu().numpy().tolist())
    e_pred = np.array(e_pred)
    e_true = np.array(e_true)
    ax.scatter(e_true, e_pred, s=8, alpha=0.5)
    lim = [
        min(e_true.min(), e_pred.min()) - 0.01,
        max(e_true.max(), e_pred.max()) + 0.01,
    ]
    ax.plot(lim, lim, 'k--', lw=1, label='perfect')
    ax.set_xlabel('True E/atom (eV)')
    ax.set_ylabel('Predicted E/atom (eV)')
    ax.set_title(title)
    mae = np.mean(np.abs(e_pred - e_true))
    ax.text(0.05, 0.93, f'MAE = {mae:.4f} eV/atom',
            transform=ax.transAxes, fontsize=9)
    ax.grid(alpha=0.3)


scatter_energy(axes[0], model_mini,       test_loader, test_device, 'MiniTrainer')
scatter_energy(axes[1], model_lgtn_eval,  test_loader, test_device, 'Lightning Trainer')
scatter_energy(axes[2], raw_fab,          test_loader, test_device, 'FabricTrainer')

plt.suptitle('Per-atom Energy: Predicted vs. True', fontsize=13)
plt.tight_layout()
plt.show()


---

## 10 · Summary

### The three trainers at a glance

| Feature | MiniTrainer | Lightning Trainer | FabricTrainer |
|---------|-------------|-------------------|---------------|
| Multi-GPU | No | Yes (DDP, FSDP) | Yes (DDP, FSDP) |
| Mixed precision | No | Yes | Yes |
| Callbacks | No | Yes (full ecosystem) | No |
| Disk checkpointing | No | Yes (automatic) | No |
| Custom loop | Yes (you own it) | No (Lightning owns it) | Yes (you own it) |
| Logging (W&B, TB) | No | Yes | Manual |
| Hydra config | No | Yes (GOALModule) | No |
| Notebook-friendly | Best | Good | Good |
| Complexity | Lowest | Medium | Medium |

### Key GOAL design principles demonstrated

1. **`AtomicGraph` is the universal data contract** — all trainers consume
   the same data object.  You never need to adapt your dataset to the trainer.

2. **Neighbour list is computed once** at dataset load time
   (`TrajectoryDataset.__init__`), not inside the training loop.  This is
   the most important performance optimisation for graph-neural-network MLIPs.

3. **Distances are recomputed inside `forward()`** from `graph.pos` so that
   autograd can trace a gradient path from energy to positions (required for
   physically correct forces).

4. **Forces are free given a differentiable energy** — $\mathbf{F}_i =
   -\nabla_i E$ via `torch.autograd.grad`.  This guarantees energy conservation.

5. **Atomic energy decomposition** ($E = \sum_i e_i$) makes the model
   size-extensive and physically sensible — a cornerstone of all modern MLIPs
   (NequIP, MACE, SchNet, PaiNN, CHGNet, …).

### Next steps

- **Real data**: Replace the LJ generator with actual DFT trajectories from
  VASP, CP2K, or FHI-aims.  `TrajectoryDataset` reads any `.traj` file ASE
  can write.  For large datasets consider `lmdb` or `hdf5` backends.
- **Production training**: Use `python src/train.py` with a Hydra YAML config
  for experiment tracking, hyperparameter sweeps, and multi-GPU runs.
- **Foundation models**: Freeze a pre-trained MACE backbone and fine-tune only
  the head.  Use `MiniTrainer` for fast iteration on the head parameters.
- **Equivariance**: Swap `InvariantGNN` for `HyperSpecModel` to get equivariant
  features — better data efficiency, especially for non-spherical systems.
- **Periodic systems / crystals**: Add `stress` to the target dict and use
  `StressLoss` in `CompositeLoss`.  Pass PBC-enabled `Atoms` objects with a
  cell to `TrajectoryDataset`.
- **Hyperparameter search**: Use the `hparams_search/` configs with Optuna,
  Ray Tune, or W&B sweeps.
